# 第一阶段「自动微分」知识地图

> 来源：《深度学习入门2：自制框架》（斋藤康毅 著，郑明智 译，人民邮电出版社 2023）
> 范围：第 1 阶段 · 步骤 01～10 · 从零构建 DeZero 框架的自动微分机制

配套 notebook：`第一阶段步骤01.ipynb` ~ `第一阶段步骤10.ipynb`（每个步骤一篇）。

## 一、总览

| 步骤 | 标题 | 新增核心 | 阶段作用 |
| --- | --- | --- | --- |
| 01 | 作为"箱子"的变量 | `Variable` 类（存 `data`） | 数据容器 |
| 02 | 创建变量的函数 | `Function` 类（`__call__`/`forward`） | 计算单元 |
| 03 | 函数的连续调用 | `Exp` 函数 + **计算图**概念 | 可组合 |
| 04 | 数值微分 | `numerical_diff`（中心差分） | 求导的过渡/对照 |
| 05 | 反向传播的理论知识 | **链式法则**（纯理论） | 数学依据 |
| 06 | 手动进行反向传播 | `grad` + `backward` + `self.input` | 求导落地 |
| 07 | 反向传播的自动化 | `creator` + `set_creator` + 递归 `backward` | 自动求导 |
| 08 | 从递归到循环 | 循环 `backward`（`funcs` + `pop`） | 高效 |
| 09 | 让函数更便利 | `square`/`exp` + `ones_like` + `as_array` | 易用 |
| 10 | 测试 | `unittest` + 梯度检验 | 可靠 |

## 二、主线脉络

整个阶段是一条清晰的递进线：

```
数据装进箱子 → 用函数加工数据 → 函数串联成计算图
     ↓              ↓                 ↓
   步骤01          步骤02            步骤03

怎么求导？先数值微分建立直觉 → 数学依据是链式法则 → 把求导写成 backward
     ↓                              ↓                    ↓
   步骤04                         步骤05               步骤06

用 creator 记录连接、自动回溯 → 递归改循环提效 → 打磨易用性 → 测试守质量
     ↓                            ↓              ↓            ↓
   步骤07                        步骤08         步骤09       步骤10
```

一句话概括：**先解决"能不能算"（01~06），再解决"能不能自动、高效、好用、可信"（07~10）。**

## 三、每步的代码增量

| 步骤 | 新增符号 | 关键点 |
| --- | --- | --- |
| 01 | `Variable.__init__(self, data)` | 把数据存进 `self.data`，变量是"箱子" |
| 02 | `Function.__call__` / `forward` | `__call__` 统一"取数据→forward→包装返回" |
| 03 | `Exp` + 计算图 | 输入输出都是 `Variable`，所以能任意串联 |
| 04 | `numerical_diff(f, x, eps=1e-4)` | 中心差分 $(f(x+h)-f(x-h))/2h$ |
| 05 | （无） | 链式法则：复合函数导数 = 各函数导数连乘 |
| 06 | `Variable.grad`、`Function.backward`、`self.input` | `backward` = 局部导数 × 传来的 `gy` |
| 07 | `Variable.creator`、`set_creator`、递归 `backward` | 变量记住"谁创造了我"，自动回溯 |
| 08 | 循环 `backward`（`funcs` 列表 + `pop`） | 等价重构，省调用栈、易扩展 |
| 09 | `square`/`exp`、`np.ones_like`、`as_array` | 三种易用性打磨 |
| 10 | `SquareTest`、`numerical_diff` 复用 | 正向/反向/梯度检验三类断言 |

## 四、核心概念速查

| 概念 | 一句话解释 |
| --- | --- |
| **Variable** | 数据的"箱子"：存 `data`（值）、`grad`（梯度）、`creator`（创造者） |
| **Function** | 计算的单元：`forward`（正向计算）+ `backward`（局部求导） |
| **计算图** | 变量与函数交替排列成的图，用来跟踪数据流向 |
| **复合函数** | 多个函数依次应用看成"一个大函数" |
| **链式法则** | 复合函数导数 = 外层导数 × 内层导数（反向传播的数学基础） |
| **backward** | 反向传播：从输出端往回，逐层乘局部导数 |
| **creator** | 变量的"父母"——创造它的那个函数，用于反向遍历 |
| **Define-by-Run** | 动态计算图：正向执行的那一刻动态建立连接 |
| **数值微分** | 用极小差分近似求导（误差大、效率低，作对照/检验用） |
| **梯度检验** | 数值微分 vs 反向传播结果对比，判断实现是否正确 |

## 五、最终形态参考（步骤09 的完整框架）

10 个步骤累积出的成果，浓缩成下面这个**可运行的代码 cell**，直接执行即可：

In [ ]:
import numpy as np

def as_array(x):
    if np.isscalar(x):
        return np.array(x)
    return x

class Variable:
    def __init__(self, data):
        if data is not None and not isinstance(data, np.ndarray):
            raise TypeError(f'{type(data)} 不是支持的 ndarray 类型')
        self.data = data
        self.grad = None
        self.creator = None

    def set_creator(self, func):
        self.creator = func

    def backward(self):
        if self.grad is None:
            self.grad = np.ones_like(self.data)
        funcs = [self.creator]
        while funcs:
            f = funcs.pop()
            x, y = f.input, f.output
            x.grad = f.backward(y.grad)
            if x.creator is not None:
                funcs.append(x.creator)

class Function:
    def __call__(self, input):
        x = input.data
        y = self.forward(x)
        output = Variable(as_array(y))
        output.set_creator(self)
        self.input = input
        self.output = output
        return output

    def forward(self, x):
        raise NotImplementedError()

    def backward(self, gy):
        raise NotImplementedError()

class Square(Function):
    def forward(self, x):
        return x ** 2
    def backward(self, gy):
        x = self.input.data
        return 2 * x * gy

class Exp(Function):
    def forward(self, x):
        return np.exp(x)
    def backward(self, gy):
        x = self.input.data
        return np.exp(x) * gy

def square(x):
    return Square()(x)

def exp(x):
    return Exp()(x)

# 使用：一句话自动求导
x = Variable(np.array(0.5))
y = square(exp(square(x)))   # y = (e^(x^2))^2
y.backward()
print(x.grad)   # 3.297442541400256

## 六、下一阶段

步骤 11 起进入**第二阶段**：扩展框架能力，如支持多输入多输出的函数、更丰富的算子，逐步逼近一个真正可用的深度学习框架。